In [ ]:
# Import libraries
import sys
sys.path.append('data_preprocessing')

from analyze_bus_stops_arrival import BusStopAnalyzer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully!")

## 1. Initialize Analyzer

In [ ]:
# Initialize the bus stop analyzer
analyzer = BusStopAnalyzer()

print(f"\n✓ Analyzer initialized")
print(f"  Total bus stops loaded: {len(analyzer.stops_df):,}")
print(f"  Total routes loaded: {len(analyzer.routes_data)}")
print(f"  Route numbers: {sorted([int(r) if r.isdigit() else r for r in analyzer.routes_data.keys()])}")

## 2. Analyze Single Date - All Routes

In [ ]:
# Analyze a specific date
analysis_date = '2025-04-01'

print(f"Analyzing date: {analysis_date}")
arrivals_df = analyzer.analyze_date(analysis_date)

# Display results
print(f"\n{'='*80}")
print("RESULTS OVERVIEW")
print(f"{'='*80}")
print(arrivals_df.info())
print(f"\n{arrivals_df.head(20)}")

## 3. Save Results

In [ ]:
# Save the analysis results
if len(arrivals_df) > 0:
    output_file = analyzer.save_results(arrivals_df, f'arrivals_{analysis_date}.csv')
    print(f"\n✓ Results saved successfully!")
else:
    print("⚠️ No arrivals data to save")

## 4. Statistical Summary

In [ ]:
if len(arrivals_df) > 0:
    print("\n" + "="*80)
    print("STATISTICAL SUMMARY")
    print("="*80)
    
    # Overall statistics
    print(f"\nTotal arrival records: {len(arrivals_df):,}")
    print(f"Unique vehicles: {arrivals_df['vehicle_id'].nunique()}")
    print(f"Unique stops visited: {arrivals_df['stop_id'].nunique()}")
    print(f"Routes analyzed: {arrivals_df['route_num'].nunique()}")
    
    # Stop duration statistics
    print(f"\nStop Duration Statistics (seconds):")
    print(f"  Mean: {arrivals_df['stop_duration_seconds'].mean():.1f}")
    print(f"  Median: {arrivals_df['stop_duration_seconds'].median():.1f}")
    print(f"  Std Dev: {arrivals_df['stop_duration_seconds'].std():.1f}")
    print(f"  Min: {arrivals_df['stop_duration_seconds'].min():.1f}")
    print(f"  Max: {arrivals_df['stop_duration_seconds'].max():.1f}")
    
    # Door events
    door_event_pct = arrivals_df['had_door_event'].sum() / len(arrivals_df) * 100
    print(f"\nDoor Events:")
    print(f"  Stops with door events: {arrivals_df['had_door_event'].sum():,} ({door_event_pct:.1f}%)")
    print(f"  Stops without door events: {(~arrivals_df['had_door_event']).sum():,} ({100-door_event_pct:.1f}%)")
    
    # By route
    print(f"\nBy Route:")
    route_summary = arrivals_df.groupby('route_num').agg({
        'vehicle_id': 'nunique',
        'stop_id': 'nunique',
        'arrival_time': 'count',
        'stop_duration_seconds': 'mean'
    }).rename(columns={
        'vehicle_id': 'Vehicles',
        'stop_id': 'Stops',
        'arrival_time': 'Arrivals',
        'stop_duration_seconds': 'Avg Duration (s)'
    }).round(1)
    print(route_summary)

## 5. Visualize Stop Duration Distribution

In [ ]:
if len(arrivals_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f'Stop Duration Analysis - {analysis_date}', fontsize=16, fontweight='bold')
    
    # 1. Histogram of stop duration
    ax1 = axes[0, 0]
    arrivals_df['stop_duration_seconds'].hist(bins=50, ax=ax1, color='#3498db', edgecolor='black')
    ax1.axvline(arrivals_df['stop_duration_seconds'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
    ax1.axvline(arrivals_df['stop_duration_seconds'].median(), color='green', linestyle='--', linewidth=2, label='Median')
    ax1.set_title('Stop Duration Distribution', fontweight='bold')
    ax1.set_xlabel('Duration (seconds)')
    ax1.set_ylabel('Frequency')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Box plot by route
    ax2 = axes[0, 1]
    arrivals_df.boxplot(column='stop_duration_seconds', by='route_num', ax=ax2)
    ax2.set_title('Stop Duration by Route', fontweight='bold')
    ax2.set_xlabel('Route Number')
    ax2.set_ylabel('Duration (seconds)')
    plt.sca(ax2)
    plt.xticks(rotation=45)
    
    # 3. Door event vs no door event
    ax3 = axes[1, 0]
    door_comparison = arrivals_df.groupby('had_door_event')['stop_duration_seconds'].mean()
    door_comparison.plot(kind='bar', ax=ax3, color=['#e74c3c', '#2ecc71'], edgecolor='black')
    ax3.set_title('Average Stop Duration: Door Event Comparison', fontweight='bold')
    ax3.set_xlabel('Had Door Event')
    ax3.set_ylabel('Avg Duration (seconds)')
    ax3.set_xticklabels(['No Door Event', 'Door Event'], rotation=0)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Arrivals by hour
    ax4 = axes[1, 1]
    arrivals_df['hour'] = pd.to_datetime(arrivals_df['arrival_time']).dt.hour
    hourly_arrivals = arrivals_df.groupby('hour').size()
    hourly_arrivals.plot(kind='bar', ax=ax4, color='#9b59b6', edgecolor='black')
    ax4.set_title('Arrivals by Hour of Day', fontweight='bold')
    ax4.set_xlabel('Hour')
    ax4.set_ylabel('Number of Arrivals')
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 6. Top Busiest Stops

In [ ]:
if len(arrivals_df) > 0:
    print("\n" + "="*80)
    print("TOP 20 BUSIEST STOPS")
    print("="*80)
    
    top_stops = arrivals_df.groupby(['stop_id', 'stop_name']).agg({
        'arrival_time': 'count',
        'vehicle_id': 'nunique',
        'stop_duration_seconds': 'mean'
    }).rename(columns={
        'arrival_time': 'Total Arrivals',
        'vehicle_id': 'Unique Vehicles',
        'stop_duration_seconds': 'Avg Duration (s)'
    }).sort_values('Total Arrivals', ascending=False).head(20)
    
    print(top_stops.round(1))
    
    # Visualize top 15
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'Top 15 Busiest Stops - {analysis_date}', fontsize=14, fontweight='bold')
    
    # Total arrivals
    ax1 = axes[0]
    top_stops.head(15)['Total Arrivals'].plot(kind='barh', ax=ax1, color='#3498db', edgecolor='black')
    ax1.set_title('By Total Arrivals', fontweight='bold')
    ax1.set_xlabel('Number of Arrivals')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Unique vehicles
    ax2 = axes[1]
    top_stops.head(15)['Unique Vehicles'].plot(kind='barh', ax=ax2, color='#e67e22', edgecolor='black')
    ax2.set_title('By Unique Vehicles', fontweight='bold')
    ax2.set_xlabel('Number of Unique Vehicles')
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## 7. Analyze Specific Route

In [ ]:
# Choose a route to analyze in detail
route_to_analyze = '1'

if len(arrivals_df) > 0 and route_to_analyze in arrivals_df['route_num'].values:
    route_df = arrivals_df[arrivals_df['route_num'] == route_to_analyze].copy()
    
    print(f"\n{'='*80}")
    print(f"ROUTE {route_to_analyze} DETAILED ANALYSIS")
    print(f"{'='*80}")
    
    print(f"\nTotal arrivals: {len(route_df):,}")
    print(f"Unique vehicles: {route_df['vehicle_id'].nunique()}")
    print(f"Unique stops: {route_df['stop_id'].nunique()}")
    
    # Stop sequence analysis
    print(f"\nStop Visit Frequency:")
    stop_freq = route_df.groupby('stop_name').size().sort_values(ascending=False).head(15)
    print(stop_freq)
    
    # Visualize route timeline for one vehicle
    sample_vehicle = route_df['vehicle_id'].iloc[0]
    vehicle_route = route_df[route_df['vehicle_id'] == sample_vehicle].sort_values('arrival_time')
    
    if len(vehicle_route) > 0:
        print(f"\nSample trip for vehicle {sample_vehicle}:")
        print(f"  Stops visited: {len(vehicle_route)}")
        print(f"  Trip start: {vehicle_route['arrival_time'].min()}")
        print(f"  Trip end: {vehicle_route['departure_time'].max()}")
        trip_duration = (pd.to_datetime(vehicle_route['departure_time'].max()) - 
                        pd.to_datetime(vehicle_route['arrival_time'].min())).total_seconds() / 60
        print(f"  Total trip duration: {trip_duration:.1f} minutes")
        
        print(f"\nStop sequence:")
        print(vehicle_route[['stop_name', 'arrival_time', 'departure_time', 'stop_duration_seconds']].to_string(index=False))
else:
    print(f"\nNo data found for route {route_to_analyze}")

## 8. Analyze Date Range (Optional)

In [ ]:
# Uncomment to analyze multiple dates
# start_date = '2025-04-01'
# end_date = '2025-04-07'
# 
# print(f"Analyzing date range: {start_date} to {end_date}")
# multi_day_arrivals = analyzer.analyze_date_range(start_date, end_date)
# 
# if len(multi_day_arrivals) > 0:
#     analyzer.save_results(multi_day_arrivals, f'arrivals_{start_date}_to_{end_date}.csv')
#     print(f"\nTotal arrivals across all dates: {len(multi_day_arrivals):,}")
#     print(f"Date range: {multi_day_arrivals['date'].min()} to {multi_day_arrivals['date'].max()}")

## 9. Export Summary for Specific Route

In [ ]:
# Create a detailed summary for a specific route
def create_route_summary(arrivals_df, route_num, output_file=None):
    """
    Create detailed route summary with average arrival times per stop
    """
    route_data = arrivals_df[arrivals_df['route_num'] == route_num].copy()
    
    if len(route_data) == 0:
        print(f"No data for route {route_num}")
        return None
    
    # Extract hour and minute from arrival time
    route_data['arrival_datetime'] = pd.to_datetime(route_data['arrival_time'])
    route_data['hour'] = route_data['arrival_datetime'].dt.hour
    route_data['minute'] = route_data['arrival_datetime'].dt.minute
    route_data['time_of_day'] = route_data['hour'] * 60 + route_data['minute']
    
    # Aggregate by stop
    summary = route_data.groupby(['stop_id', 'stop_name']).agg({
        'arrival_time': 'count',
        'time_of_day': ['mean', 'std', 'min', 'max'],
        'stop_duration_seconds': ['mean', 'std'],
        'had_door_event': 'sum'
    }).round(2)
    
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary = summary.rename(columns={
        'arrival_time_count': 'total_visits',
        'time_of_day_mean': 'avg_arrival_minute',
        'time_of_day_std': 'std_arrival_minute',
        'time_of_day_min': 'earliest_minute',
        'time_of_day_max': 'latest_minute',
        'stop_duration_seconds_mean': 'avg_duration_sec',
        'stop_duration_seconds_std': 'std_duration_sec',
        'had_door_event_sum': 'door_events'
    })
    
    # Convert minutes to HH:MM format
    def minutes_to_time(minutes):
        h = int(minutes // 60)
        m = int(minutes % 60)
        return f"{h:02d}:{m:02d}"
    
    summary['avg_arrival_time'] = summary['avg_arrival_minute'].apply(minutes_to_time)
    summary['earliest_time'] = summary['earliest_minute'].apply(minutes_to_time)
    summary['latest_time'] = summary['latest_minute'].apply(minutes_to_time)
    
    if output_file:
        summary.to_csv(output_file)
        print(f"Saved route {route_num} summary to: {output_file}")
    
    return summary

# Example usage
if len(arrivals_df) > 0:
    route_summary = create_route_summary(arrivals_df, '1', 
                                         'Bus_route_data/arrival_times/route_1_summary.csv')
    print("\nRoute Summary (first 10 stops):")
    print(route_summary[['total_visits', 'avg_arrival_time', 'avg_duration_sec', 'door_events']].head(10))

## Summary

This notebook provides comprehensive analysis of:

1. **Door Event Detection** - Identifies when bus doors open/close
2. **GPS-to-Stop Matching** - Maps GPS coordinates to bus stops
3. **Arrival Time Calculation** - Determines arrival and departure times
4. **Route Analysis** - Identifies routes and trip patterns
5. **Statistical Analysis** - Stop duration, frequency, patterns

### Output Files:
- `arrivals_YYYY-MM-DD.csv` - Detailed arrival records
- `arrivals_YYYY-MM-DD_summary.txt` - Statistical summary
- `route_X_summary.csv` - Per-route summary with average times

### Next Steps:
- Use arrival data for real-time predictions
- Integrate with GTFS schedule data
- Build delay/reliability metrics
- Create passenger information displays